# Waterfilling Levels

In [59]:
import sys
sys.path.insert(1, '../../functions_multi_resource')
import importlib
import numpy as np
import nbformat
import plotly.express
import plotly.express as px
import pandas as pd
from scipy.optimize import minimize
import scipy
import matplotlib.pyplot as plt
import seaborn as sns
from food_bank_functions import *
import time
# importlib.reload(food_bank_functions)

In [2]:
np.random.seed(1)

In [3]:
n = 3
k = 4
size = [1., 2., 3.]
B = np.zeros((k, n*k))
for i in range(n):
    B[:,k*i:k*(i+1)] = size[i]*np.eye(k)
print(B)

[[1. 0. 0. 0. 2. 0. 0. 0. 3. 0. 0. 0.]
 [0. 1. 0. 0. 0. 2. 0. 0. 0. 3. 0. 0.]
 [0. 0. 1. 0. 0. 0. 2. 0. 0. 0. 3. 0.]
 [0. 0. 0. 1. 0. 0. 0. 2. 0. 0. 0. 3.]]


# Generating Distribution

In [4]:
n = 6
k = 9

#### Different Population of Locations

In [5]:
# size = [1, 1, 1, 1, 1, 1]
size = [928, 1200, 420, 429, 103, 393]
size = size / np.sum(size) * 100
county = ['Broome', 'Steuben', 'Chemung', 'Tioga', 'Schuyler', 'Tompkins']
print(county)
print(size)

['Broome', 'Steuben', 'Chemung', 'Tioga', 'Schuyler', 'Tompkins']
[26.72041463 34.55226029 12.0932911  12.35243305  2.96573568 11.31586525]


#### Distribution on Weights

In [6]:
product = ['cereal', 'diapers', 'pasta', 'paper', 'prepared_meals', 'rice', 'meat', 'fruit', 'produce']
w = [3.9, 3.5, 3.2, 3, 2.8, 2.7, 1.9, 1.2, .2]
print(product)
print(w)
budget = np.sum(w)*np.ones(k)
print(budget)

['cereal', 'diapers', 'pasta', 'paper', 'prepared_meals', 'rice', 'meat', 'fruit', 'produce']
[3.9, 3.5, 3.2, 3, 2.8, 2.7, 1.9, 1.2, 0.2]
[22.4 22.4 22.4 22.4 22.4 22.4 22.4 22.4 22.4]


In [7]:
w_1 = [1, 0, 1, 0, 0, 1, 1, 1, 1] # soup kitchen 
w_2 = [1, 1, 1, 1, 1, 1, 1, 1, 1] # general warehouse
w_3 = np.random.randint(0,2,9)
w_4 = np.random.randint(0,2,9)
w_5 = np.random.randint(0,2,9)
w_6 = np.random.randint(0,2,9)
w_7 = np.random.randint(0,2,9)
w_8 = np.random.randint(0,2,9)
weight_matrix = np.asarray([ w_1, w_2, w_3, w_4, w_5, w_6, w_7, w_8])
print(weight_matrix)
weight_distribution = [1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/8]

[[1 0 1 0 0 1 1 1 1]
 [1 1 1 1 1 1 1 1 1]
 [1 1 0 0 1 1 1 1 1]
 [0 0 1 0 1 1 0 0 1]
 [0 0 0 1 0 0 1 0 0]
 [0 1 0 0 0 1 1 1 1]
 [1 0 0 0 1 1 1 1 1]
 [1 0 1 1 0 0 1 0 0]]


In [8]:
expected_weights = np.zeros((n,k))

In [9]:
for i in range(n):
    for j in range(k):
#         print(i,j)
        expected_weights[i,j] = w[j] * (1/8) * (w_1[j] + w_2[j] + w_3[j] + w_4[j] + w_5[j] + w_6[j] + w_7[j]+w_8[j])
print(expected_weights)

[[2.4375 1.3125 1.6    1.125  1.4    2.025  1.6625 0.75   0.15  ]
 [2.4375 1.3125 1.6    1.125  1.4    2.025  1.6625 0.75   0.15  ]
 [2.4375 1.3125 1.6    1.125  1.4    2.025  1.6625 0.75   0.15  ]
 [2.4375 1.3125 1.6    1.125  1.4    2.025  1.6625 0.75   0.15  ]
 [2.4375 1.3125 1.6    1.125  1.4    2.025  1.6625 0.75   0.15  ]
 [2.4375 1.3125 1.6    1.125  1.4    2.025  1.6625 0.75   0.15  ]]


In [10]:
size

array([26.72041463, 34.55226029, 12.0932911 , 12.35243305,  2.96573568,
       11.31586525])

In [11]:
x, sol = solve(expected_weights, n, k, budget, size)

In [12]:
x = np.reshape(x, (n,k))

In [13]:
print(x)

[[0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224]
 [0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224]
 [0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224]
 [0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224]
 [0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224]
 [0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224 0.224]]


In [14]:
print(excess(x, budget, size))
print(envy_utility(x, expected_weights))
print(proportionality_utility(x, expected_weights, size, budget))

[1.18423789e-15 1.18423789e-15 1.18423789e-15 1.18423789e-15
 1.18423789e-15 1.18423789e-15 1.18423789e-15 1.18423789e-15
 1.18423789e-15]
[0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0.]


In [15]:
x.shape

(6, 9)

In [16]:
print(n)

6


### Test

In [17]:
obs_types = np.random.randint(0,8,n)
print(obs_types)

observed_weights = np.zeros((n, k))
for i in range(n):
    observed_weights[i,:] = np.multiply(weight_matrix[obs_types[i], :], w)
print(observed_weights)

[5 3 7 2 1 0]
[[0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 3.5 3.2 3.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  3.2 0.  0.  2.7 1.9 1.2 0.2]]


In [18]:
opt, _ = solve(observed_weights, n, k, budget, size)
opt = np.reshape(opt, (n,k))
print(np.around(opt, decimals=4))

[[0.0000e+00 7.5900e-01 0.0000e+00 0.0000e+00 0.0000e+00 6.2700e-02
  6.5460e-01 7.3140e-01 3.5230e-01]
 [0.0000e+00 0.0000e+00 6.0780e-01 0.0000e+00 5.7240e-01 5.1950e-01
  0.0000e+00 0.0000e+00 3.3620e-01]
 [1.5800e-01 0.0000e+00 2.1000e-03 1.4497e+00 0.0000e+00 0.0000e+00
  2.4000e-02 0.0000e+00 0.0000e+00]
 [9.0190e-01 1.6730e-01 0.0000e+00 0.0000e+00 2.1240e-01 1.2900e-02
  1.0100e-01 6.4900e-02 7.6500e-02]
 [3.3000e-03 1.8100e-02 0.0000e+00 1.6415e+00 0.0000e+00 5.0000e-04
  6.5000e-03 2.8000e-03 3.0000e-04]
 [8.2530e-01 0.0000e+00 1.2130e-01 0.0000e+00 0.0000e+00 2.3100e-01
  2.9610e-01 1.8100e-01 3.7300e-02]]


../../functions_multi_resource\food_bank_functions.py:20: RuntimeWarning: divide by zero encountered in log
  value[i] = np.log(np.dot(X[i,:], W[i,:]))


In [19]:
print(excess(opt, budget, size))
print(envy_utility(opt, observed_weights))
print(proportionality_utility(opt, observed_weights, size, budget))

[ 1.62181379e-12  4.13239813e-12  1.04212935e-13 -4.31654712e-13
  7.32451137e-13  3.61488617e-12  2.60177065e-12  1.17357975e-12
  1.61648472e-13]
[-3.54764534e+00 -3.99814988e+00 -6.81331137e-02 -8.90894903e-05
  1.05736095e-04 -1.18011486e+00]
[-2.88957268e+00 -3.02399025e+00 -2.32960264e+00 -1.38886177e+00
  4.39689694e-05 -2.08323558e+00]


In [20]:
allocation = et_full(expected_weights, observed_weights, n, k, budget, size)

In [21]:
budget_used = np.zeros(k)
for i in range(n):
    budget_used += size[i] * allocation[i,:]

In [22]:
budget

array([22.4, 22.4, 22.4, 22.4, 22.4, 22.4, 22.4, 22.4, 22.4])

In [23]:
budget - budget_used

array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        8.92989421, 22.4       ,  0.        , 10.50768364])

In [24]:
allocation.shape

(6, 9)

In [25]:
np.sum(allocation, axis=0)

array([1.87045025e+00, 8.38310345e-01, 6.48293333e-01, 1.85226667e+00,
       6.48293333e-01, 5.75248755e-01, 4.21191196e-12, 8.38310345e-01,
       5.01060404e-01])

In [26]:
allocation[:,j]

array([1.05897611e-01, 1.97581396e-01, 3.05916161e-12, 1.75040583e-13,
       1.28056149e-13, 1.97581396e-01])

In [27]:
true_alloc = np.zeros(allocation.shape[1])

In [28]:
for j in range(k):
    true_alloc[j] += np.dot(size, allocation[:,j])


In [29]:
true_alloc

array([2.24000000e+01, 2.24000000e+01, 2.24000000e+01, 2.24000000e+01,
       2.24000000e+01, 1.34701058e+01, 7.54798603e-11, 2.24000000e+01,
       1.18923164e+01])

In [30]:
print(size)

[26.72041463 34.55226029 12.0932911  12.35243305  2.96573568 11.31586525]


In [31]:
np.sum(true_alloc, axis=0)

159.7624221543077

In [32]:
print(budget)

[22.4 22.4 22.4 22.4 22.4 22.4 22.4 22.4 22.4]


In [33]:
budget - np.sum(true_alloc, axis=0)

array([-137.36242215, -137.36242215, -137.36242215, -137.36242215,
       -137.36242215, -137.36242215, -137.36242215, -137.36242215,
       -137.36242215])

In [34]:
budget

array([22.4, 22.4, 22.4, 22.4, 22.4, 22.4, 22.4, 22.4, 22.4])

In [35]:
print(np.around(allocation, decimals=5))

[[0.      0.83831 0.      0.      0.      0.07324 0.      0.83831 0.1059 ]
 [0.      0.      0.64829 0.      0.64829 0.251   0.      0.      0.19758]
 [0.06165 0.      0.      1.85227 0.      0.      0.      0.      0.     ]
 [1.48649 0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.04246 0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.27985 0.      0.      0.      0.      0.251   0.      0.      0.19758]]


In [36]:
print(excess(allocation, budget, size))
print(envy_utility(allocation, observed_weights))
print(proportionality_utility(allocation, observed_weights, size, budget))

[0.         0.         0.         0.         0.         1.4883157
 3.73333333 0.         1.75128061]
[-3.44176054e+00 -3.88976000e+00  4.69631658e-05 -1.63830412e+00
  5.63170286e+00  3.98863463e+00]
[-2.03098781 -2.61338727 -3.10924497 -2.16849193  4.85201093  1.1257427 ]


In [37]:
print(np.max(allocation - opt))

0.5845435154839835


In [38]:
observed_weights[3,:]

array([3.9, 3.5, 0. , 0. , 2.8, 2.7, 1.9, 1.2, 0.2])

In [39]:
allocation = et_online(expected_weights, observed_weights, n, k, budget, size)

c:\users\seanr\anaconda3\envs\food-bank\lib\site-packages\scipy\optimize\_constraints.py:422: OptimizeWarning: Equality and inequality constraints are specified in the same element of the constraint list. For efficient use with this method, equality and inequality constraints should be specified in separate elements of the constraint list. 
  "in separate elements of the constraint list. ", OptimizeWarning)


In [40]:
print(np.around(allocation, decimals=5))

[[0.      0.83831 0.      0.      0.      0.07324 0.      0.83831 0.1059 ]
 [0.      0.      0.64829 0.      0.64829 0.29316 0.      0.      0.44514]
 [0.22095 0.      0.      1.85227 0.      0.      0.      0.      0.     ]
 [1.5971  0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.      0.      2.00283 0.      0.      0.38823]
 [0.      0.      0.      0.      0.      0.38653 0.      0.      0.2685 ]]


In [41]:
print(excess(allocation, budget, size))
print(envy_utility(allocation, observed_weights))
print(proportionality_utility(allocation, observed_weights, size, budget))

[5.92118946e-16 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 3.73333333e+00 0.00000000e+00
 0.00000000e+00]
[ 1.32629155  0.71497    -0.18981999 -0.74339486  0.93321484  5.13134209]
[-2.03098781 -2.77670935 -3.7304942  -2.59987421 -0.46767935  1.83706788]


In [42]:
alloc = hope_online(weight_matrix, np.asarray(weight_distribution), obs_types, n, k, budget, size)

In [43]:
print(np.around(alloc, decimals=10))

[[0.         0.52353905 0.         0.         0.         0.36042999
  0.28199544 0.48959349 0.36042999]
 [0.         0.         0.5686264  0.         0.5686264  0.32414653
  0.         0.         0.32414653]
 [0.71522766 0.         0.00382638 0.99068035 0.         0.
  0.51188658 0.         0.        ]
 [0.76930863 0.5252264  0.         0.         0.03520615 0.03102894
  0.3803011  0.44947141 0.03102894]
 [0.17063527 0.12175012 0.11253788 1.47353997 0.08964578 0.0225755
  0.07593704 0.1324563  0.0225755 ]
 [0.33065535 0.13802893 0.20967388 0.53458432 0.1813317  0.09888153
  0.331545   0.2980738  0.09888153]]


In [44]:
print(excess(allocation, budget, size))
print(envy_utility(allocation, observed_weights))
print(proportionality_utility(allocation, observed_weights, size, budget))

[5.92118946e-16 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 3.73333333e+00 0.00000000e+00
 0.00000000e+00]
[ 1.32629155  0.71497    -0.18981999 -0.74339486  0.93321484  5.13134209]
[-2.03098781 -2.77670935 -3.7304942  -2.59987421 -0.46767935  1.83706788]


In [45]:
alloc = hope_full(weight_matrix, np.asarray(weight_distribution), obs_types, n, k, budget, size)

In [46]:
print(np.around(alloc, decimals=3))

[[0.    0.524 0.    0.    0.    0.36  0.282 0.49  0.36 ]
 [0.    0.    0.552 0.    0.536 0.37  0.    0.    0.37 ]
 [0.573 0.    0.177 1.083 0.    0.    0.183 0.    0.   ]
 [1.029 0.253 0.    0.    0.092 0.    0.28  0.254 0.   ]
 [0.076 0.106 0.056 1.296 0.097 0.    0.08  0.106 0.   ]
 [0.224 0.    0.091 0.    0.    0.    0.335 0.28  0.   ]]


In [47]:
print(excess(allocation, budget, size))
print(envy_utility(allocation, observed_weights))
print(proportionality_utility(allocation, observed_weights, size, budget))

[5.92118946e-16 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 0.00000000e+00 3.73333333e+00 0.00000000e+00
 0.00000000e+00]
[ 1.32629155  0.71497    -0.18981999 -0.74339486  0.93321484  5.13134209]
[-2.03098781 -2.77670935 -3.7304942  -2.59987421 -0.46767935  1.83706788]


In [48]:
allocation = hope_full_v2(weight_matrix, np.asarray(weight_distribution), obs_types, n, k, budget, size)

../../functions_multi_resource\food_bank_functions.py:63: RuntimeWarning: divide by zero encountered in log
  value[i] = np.sum([weight_distribution[i,j] * np.log(np.dot(X[i,:], weight_matrix[j,:])) for j in range(num_types)])
../../functions_multi_resource\food_bank_functions.py:63: RuntimeWarning: invalid value encountered in double_scalars
  value[i] = np.sum([weight_distribution[i,j] * np.log(np.dot(X[i,:], weight_matrix[j,:])) for j in range(num_types)])


In [49]:
print(np.around(allocation, decimals=5))

[[0.      0.83831 0.      0.      0.      0.05497 0.      0.83831 0.05497]
 [0.      0.      0.64829 0.      0.64829 0.25233 0.      0.      0.25233]
 [0.93846 0.      0.      1.85227 0.      0.      0.      0.      0.     ]
 [0.89464 0.      0.      0.      0.      0.06286 0.05605 0.      0.06286]
 [0.      0.      0.      0.      0.      0.08711 0.12299 0.      0.08711]
 [0.      0.      0.      0.      0.      0.21721 0.33523 0.      0.21721]]


In [50]:
print(excess(alloc, budget, size))
print(envy_utility(alloc, observed_weights))
print(proportionality_utility(alloc, observed_weights, size, budget))

[-5.92118946e-16  8.27365676e-01  0.00000000e+00  9.10234452e-01
  4.10061268e-01  0.00000000e+00  8.58717923e-01  4.48475943e-01
  0.00000000e+00]
[-2.27614018 -3.29230793 -1.85075378 -1.99556288  1.11180319  2.71201623]
[-1.87293717 -2.3439549  -3.70906073 -2.36770005 -0.26765753  0.79518449]


In [51]:
allocation = hope_online_v2(weight_matrix, np.asarray(weight_distribution), obs_types, n, k, budget, np.asarray(size))

In [52]:
print(np.around(allocation, decimals=5))

[[0.      0.83831 0.      0.      0.      0.05497 0.      0.83831 0.05497]
 [0.      0.      0.64829 0.      0.64829 0.36348 0.      0.      0.36348]
 [1.85227 0.      0.      0.88351 0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.      0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.65033 0.      0.      0.      0.      0.     ]
 [0.      0.      0.      0.86487 0.      0.73987 1.97952 0.      0.73987]]


In [53]:
print(excess(alloc, budget, size))
print(envy_utility(alloc, observed_weights))
print(proportionality_utility(alloc, observed_weights, size, budget))

[-5.92118946e-16  8.27365676e-01  0.00000000e+00  9.10234452e-01
  4.10061268e-01  0.00000000e+00  8.58717923e-01  4.48475943e-01
  0.00000000e+00]
[-2.27614018 -3.29230793 -1.85075378 -1.99556288  1.11180319  2.71201623]
[-1.87293717 -2.3439549  -3.70906073 -2.36770005 -0.26765753  0.79518449]


## Experiment

In [61]:
group = np.arange(n)
num_iterations = 100

budget = [np.sum(w) for j in range(k)]


# 8 different algorithms
run_time = np.zeros((5, num_iterations))
env = np.zeros((8,num_iterations))
po = np.zeros((8,num_iterations))
prop = np.zeros((8,num_iterations))
linf = np.zeros((8,num_iterations))
l1 = np.zeros((8, num_iterations))
max_min = np.zeros((8, num_iterations))
for i in range(num_iterations):
    print(i)
    obs_types = np.random.randint(0,8,n)
    print(obs_types)

    observed_weights = np.zeros((n, k))
    for i in range(n):
        observed_weights[i,:] = np.multiply(weight_matrix[obs_types[i], :], w)
    print(observed_weights)

    start = time.perf_counter()
    opt, _ = solve(observed_weights, n, k, budget, size)
    run_time[0,i] = time.perf_counter() - start
    
    opt = np.reshape(opt, (n,k))
    
    start = time.perf_counter()
    et_full_alloc = et_full(expected_weights, observed_weights, n, k, budget, size)
    run_time[1,i] = time.perf_counter() - start
    
    start = time.perf_counter()
    et_online_alloc = et_online(expected_weights, observed_weights, n, k, budget, size)
    run_time[2,i] = time.perf_counter() - start
    
    prop_alloc = proportional_alloc(n, k, np.asarray(budget), size)
    
    start = time.perf_counter()
    weight_full_alloc = hope_full(weight_matrix, np.asarray(weight_distribution), obs_types, n, k, budget, size)
    run_time[3,i] = time.perf_counter() - start
    
    start = time.perf_counter()
    hope_online_alloc = hope_online(weight_matrix, np.asarray(weight_distribution), obs_types, n, k, budget, size)
    run_time[4,i] = time.perf_counter() - start
    
    
    hope_full_v2_alloc = hope_full_v2(weight_matrix, np.asarray(weight_distribution), obs_types, n, k, budget, size)
    hope_online_v2_alloc = hope_online_v2(weight_matrix, np.asarray(weight_distribution), obs_types, n, k, budget, size)
    offline = offline_alloc(weight_matrix, weight_distribution, n, k, budget, size)
    
    # comparing proportional_allocation
    env[0,i] = np.max(envy_utility(prop_alloc, observed_weights))
    po[0,i] = np.max(excess(prop_alloc, budget, size))
    prop[0,i] = np.amax(proportionality_utility(prop_alloc, observed_weights, size, budget))
    linf[0,i] = np.amax(np.abs(opt - prop_alloc))    
    l1[0,i] = np.sum(np.abs(opt - prop_alloc)) 
    
    # comparing et_online

    env[1,i] = np.max(envy_utility(et_online_alloc, observed_weights))
    po[1,i] = np.max(excess(et_online_alloc, budget, size))
    prop[1,i] = np.amax(proportionality_utility(et_online_alloc, observed_weights, size, budget))
    linf[1,i] = np.amax(np.abs(opt - et_online_alloc))
    l1[1,i] = np.sum(np.abs(opt - et_online_alloc))
    
    # comparing et_full

    env[2,i] = np.max(envy_utility(et_full_alloc, observed_weights))
    po[2,i] = np.max(excess(et_full_alloc, budget, size))
    prop[2,i] = np.amax(proportionality_utility(et_full_alloc, observed_weights, size, budget))
    linf[2,i] = np.amax(np.abs(opt - et_full_alloc))    
    l1[2,i] = np.sum(np.abs(opt - et_full_alloc))
    
    # comparing hope_online

    env[3,i] = np.max(envy_utility(hope_online_alloc, observed_weights))
    po[3,i] = np.max(excess(hope_online_alloc, budget, size))
    prop[3,i] = np.amax(proportionality_utility(hope_online_alloc, observed_weights, size, budget))
    linf[3,i] = np.amax(np.abs(opt - hope_online_alloc)) 
    l1[3,i] = np.sum(np.abs(opt - hope_online_alloc))
    
    
    # comparing hope_full
    

    env[4,i] = np.max(envy_utility(weight_full_alloc, observed_weights))
    po[4,i] = np.max(excess(weight_full_alloc, budget, size))
    prop[4,i] = np.amax(proportionality_utility(weight_full_alloc, observed_weights, size, budget))
    linf[4,i] = np.amax(np.abs(opt - weight_full_alloc))
    l1[4, i] = np.sum(np.abs(opt - weight_full_alloc))
    
    # comparing hope_online_v2
    env[5,i] = np.max(envy_utility(hope_online_v2_alloc, observed_weights))
    po[5,i] = np.max(excess(hope_online_v2_alloc, budget, size))
    prop[5,i] = np.amax(proportionality_utility(hope_online_v2_alloc, observed_weights, size, budget))
    linf[5,i] = np.amax(np.abs(opt - hope_online_v2_alloc)) 
    l1[6, i] = np.sum(np.abs(opt - hope_online_v2_alloc))
    
    # comparing hope_full_v2
    env[6,i] = np.max(envy_utility(hope_full_v2_alloc, observed_weights))
    po[6,i] = np.max(excess(hope_full_v2_alloc, budget, size))
    prop[6,i] = np.amax(proportionality_utility(hope_full_v2_alloc, observed_weights, size, budget))
    linf[6,i] = np.amax(np.abs(opt - hope_full_v2_alloc)) 
    l1[7,i] = np.sum(np.abs(opt - hope_full_v2_alloc))
    
    # comparing offline
    env[7,i] = np.max(envy_utility(offline, observed_weights))
    po[7,i] = np.max(excess(offline, budget, size))
    prop[7,i] = np.amax(proportionality_utility(offline, observed_weights, size, budget))
    linf[7,i] = np.amax(np.abs(opt - offline))
    l1[7,i] = np.sum(np.abs(opt - offline))          
    
    

0
[3 4 7 3 3 4]
[[0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]]
1
[0 6 7 5 3 5]
[[3.9 0.  3.2 0.  0.  2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]]
2
[1 4 7 4 3 2]
[[3.9 3.5 3.2 3.  2.8 2.7 1.9 1.2 0.2]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]]
3
[2 2 6 2 6 6]
[[3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.

33
[4 4 4 5 0 1]
[[0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [3.9 0.  3.2 0.  0.  2.7 1.9 1.2 0.2]
 [3.9 3.5 3.2 3.  2.8 2.7 1.9 1.2 0.2]]
34
[0 7 7 6 1 2]
[[3.9 0.  3.2 0.  0.  2.7 1.9 1.2 0.2]
 [3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 3.5 3.2 3.  2.8 2.7 1.9 1.2 0.2]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]]
35
[7 2 4 5 5 6]
[[3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]]
36
[7 2 4 5 6 0]
[[3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  3.2 0.  0.  2.7 1.9 1.

66
[7 1 7 1 6 6]
[[3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [3.9 3.5 3.2 3.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [3.9 3.5 3.2 3.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]]
67
[5 3 5 5 4 4]
[[0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]]
68
[2 5 6 2 2 5]
[[3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [3.9 0.  0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]]
69
[2 7 3 4 1 3]
[[3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 0.  3.2 3.  0.  0.  1.9 0.  0. ]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [0.  0.  0.  3.  0.  0.  1.9 0.  0. ]
 [3.9 3.5 3.2 3.  2.8 2.7 1.9 1.2 0.2]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.

99
[5 3 2 2 5 1]
[[0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [0.  0.  3.2 0.  2.8 2.7 0.  0.  0.2]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [3.9 3.5 0.  0.  2.8 2.7 1.9 1.2 0.2]
 [0.  3.5 0.  0.  0.  2.7 1.9 1.2 0.2]
 [3.9 3.5 3.2 3.  2.8 2.7 1.9 1.2 0.2]]


In [55]:
env_std = np.std(env, axis=1)
po_std = np.std(po, axis=1)
prop_std = np.std(po, axis=1)
linf_std = np.std(linf, axis=1)
l1_std = np.std(l1, axis=1)

In [56]:
print(env_std)

[0.         0.33178885 0.24258256 0.50476062 0.10814612 0.73544914
 0.60744741 0.        ]


In [57]:
env = np.average(env,axis=1)
po = np.average(po,axis=1)
prop = np.average(prop,axis=1)
linf = np.average(linf,axis=1)
l1 = np.average(l1, axis=1)

In [58]:
print('proportional, et_online, et_full, hope_online, hope_full, hope_online_v2, hope_full_v2, offline')
print('envy:')
print(env)
print('po')
print(np.around(po, decimals=5))
print('prop')
print(prop)
print('sum')
print(env+po+prop)
print('linf')
print(linf)
print('l1')
print(l1)

proportional, et_online, et_full, hope_online, hope_full, hope_online_v2, hope_full_v2, offline
envy:
[0.         0.03334603 0.02438046 0.05073035 0.01086909 0.07391542
 0.06105076 0.        ]
po
[0.      0.03733 0.03733 0.03733 0.03311 0.      0.03311 0.     ]
prop
[ 0.          0.00744211  0.00717549  0.00209715 -0.00589069  0.036288
  0.02688905  0.        ]
sum
[1.18423789e-17 7.81214825e-02 6.88892864e-02 9.01608342e-02
 3.80871471e-02 1.10203420e-01 1.21048557e-01 1.18423789e-17]
linf
[0.00678184 0.01086077 0.00571236 0.00511068 0.00439237 0.01958926
 0.00902184 0.00678184]
l1
[0.10065933 0.09861758 0.08486595 0.0736642  0.05067532 0.
 0.17130939 0.10065933]


In [62]:
print(np.average(run_time, axis=1))

[0.00026003 0.00154687 0.0008037  0.00403043 0.00296563]
